# 01 - Big Data Storage and Processing

CSE-CIC-IDS2018, 16,233,002 flows, 6.5 GB across 10 daily CSVs.

Builds a bronze/silver/gold lakehouse and takes the storage and processing
measurements. Model training is in notebook 02.

Written as an iterative record - what was tried, what broke, what the numbers say.

## 0. Setup

In [ ]:
import json, os, shutil, time, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings("ignore")

PROJECT = Path(r"D:\Saad A2")
DATA    = PROJECT / "datasets"
LAKE    = PROJECT / "lakehouse"
BRONZE  = LAKE / "bronze"
SILVER  = LAKE / "silver"
GOLD    = LAKE / "gold"
RESULTS = PROJECT / "results"
for p in (LAKE, RESULTS):
    p.mkdir(exist_ok=True)

# 9 files have 80 cols, this one has 84 (see section 2). Typo is the dataset's own.
WIDE_FILE   = "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv"
WIDE_EXTRAS = ["Flow ID", "Src IP", "Src Port", "Dst IP"]

# dropped under P2/P3, kept under P1 so the effect gets measured
LEAKY = ["Dst Port", "Timestamp"] + WIDE_EXTRAS

BENIGN    = "Benign"
ZERO_DAY  = "Bot"                      # held out of training under P3
TS_FORMAT = "dd/MM/yyyy HH:mm:ss"      # day first, not US format

CSV_FILES = sorted(DATA.glob("*.csv"))
print(f"{len(CSV_FILES)} files, {sum(f.stat().st_size for f in CSV_FILES)/1024**3:.2f} GB")
for f in CSV_FILES:
    print(f"  {f.stat().st_size/1024**2:8.1f} MB  {f.name}")

10 files, 6.41 GB
     336.0 MB  Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
     318.3 MB  Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
     365.1 MB  Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
    3867.1 MB  Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
     102.8 MB  Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
     358.5 MB  Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
     364.9 MB  Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
     341.6 MB  Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
     313.7 MB  Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
     199.6 MB  Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv


## 1. Iteration 0 - single node baseline

Need to show pandas actually can't handle this, not just claim it.
Load one file, measure, extrapolate, then try the full load.

Spark is deliberately NOT started yet. The JVM would reserve 8 GB up front and
the pandas test would then be competing against it, which would not be a fair
measurement. Restart the kernel before running this section if Spark has
already been started in this session.

In [ ]:
# 1.0 fair-test check. This measurement is only meaningful if the machine is
# not already loaded. Close other apps if free memory is low.
import psutil
free0  = psutil.virtual_memory().available / 1024**3
total0 = psutil.virtual_memory().total / 1024**3
print(f"RAM {total0:.1f} GB total, {free0:.1f} GB free ({100*free0/total0:.0f}%)")
if free0 < 2.5:
    print("\nWARNING: less than half of RAM is free.")
    print("The baseline below will understate what pandas could do on an idle machine.")
    print("Restart the kernel, close other applications, and re-run this section.")
else:
    print("ok - enough headroom for a fair baseline")

RAM 7.9 GB total, 3.4 GB free (43%)
ok - enough headroom for a fair baseline


In [ ]:
# 1.1 one file
single = DATA / "Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv"   # smallest, 103 MB

t0 = time.perf_counter()
df_single = pd.read_csv(single, low_memory=False)
t_single = time.perf_counter() - t0

mem_single  = df_single.memory_usage(deep=True).sum() / 1024**3
disk_single = single.stat().st_size / 1024**3

print(f"on disk       : {disk_single:.2f} GB")
print(f"rows x cols   : {df_single.shape[0]:,} x {df_single.shape[1]}")
print(f"load time     : {t_single:.1f} s")
print(f"in memory     : {mem_single:.2f} GB")
print(f"multiplier    : {mem_single/disk_single:.1f}x disk")

on disk       : 0.10 GB
rows x cols   : 331,125 x 80
load time     : 8.8 s
in memory     : 1.48 GB
multiplier    : 14.8x disk


In [ ]:
# 1.2 extrapolate to all 10
disk_total     = sum(f.stat().st_size for f in CSV_FILES) / 1024**3
projected_mem  = disk_total * (mem_single / disk_single)
projected_time = t_single * (disk_total / disk_single)
available      = psutil.virtual_memory().available / 1024**3
total_ram      = psutil.virtual_memory().total / 1024**3

print(f"corpus on disk      : {disk_total:.2f} GB")
print(f"projected memory    : {projected_mem:.1f} GB")
print(f"projected load time : {projected_time/60:.1f} min")
print(f"machine RAM         : {total_ram:.1f} GB  ({available:.1f} GB free)")
print(f"verdict: {'exceeds available memory' if projected_mem > available else 'fits, check timing'}")

legacy_projection = dict(
    disk_gb=round(disk_total, 2),
    projected_mem_gb=round(projected_mem, 1),
    projected_load_min=round(projected_time / 60, 1),
    machine_ram_gb=round(total_ram, 1),
    available_gb=round(available, 1),
    single_file_rows=int(df_single.shape[0]),
    memory_multiplier=round(mem_single / disk_single, 1),
)
del df_single

corpus on disk      : 6.41 GB
projected memory    : 94.6 GB
projected load time : 9.3 min
machine RAM         : 7.9 GB  (3.2 GB free)
verdict: exceeds available memory


### 1.3 Full load attempt

Loads in chunks rather than whole files, so the result is the actual row ceiling
rather than just "could not start". Stops before the machine begins swapping.

In [ ]:
# 1.3 chunked load until memory runs out
SAFETY_GB  = 0.7          # stop before the OS starts swapping
CHUNK_ROWS = 100_000

frames, loaded_rows, stopped_at = [], 0, None
t0 = time.perf_counter()

for i, f in enumerate(CSV_FILES, 1):
    if stopped_at:
        break
    for chunk in pd.read_csv(f, low_memory=False, chunksize=CHUNK_ROWS):
        free = psutil.virtual_memory().available / 1024**3
        if free < SAFETY_GB:
            stopped_at = f"{f.name} after {loaded_rows:,} rows"
            print(f"\nout of memory in file {i}/{len(CSV_FILES)} ({f.name})")
            print(f"  {free:.2f} GB free, floor {SAFETY_GB} GB")
            break
        try:
            frames.append(chunk)
            loaded_rows += len(chunk)
        except MemoryError:
            stopped_at = f"{f.name} MemoryError at {loaded_rows:,} rows"
            break
    print(f"  [{i:2}/{len(CSV_FILES)}] {f.name[:36]:36} "
          f"cumulative {loaded_rows:>10,} rows | "
          f"{psutil.virtual_memory().available/1024**3:5.2f} GB free")

t_legacy = time.perf_counter() - t0
pct = 100 * loaded_rows / 16_233_002
print(f"\npandas held {loaded_rows:,} of 16,233,002 rows ({pct:.1f}%) "
      f"in {t_legacy/60:.1f} min")
print(f"stopped at: {stopped_at or 'completed'}")

legacy_projection.update(
    rows_loaded_single_node=int(loaded_rows),
    pct_of_corpus=round(pct, 1),
    stopped_at=stopped_at,
    elapsed_min=round(t_legacy / 60, 1),
    chunk_rows=CHUNK_ROWS,
)
del frames
import gc; gc.collect()
print(f"memory released, {psutil.virtual_memory().available/1024**3:.2f} GB free")

  [ 1/10] Friday-02-03-2018_TrafficForML_CICFl cumulative  1,048,575 rows |  2.92 GB free
  [ 2/10] Friday-16-02-2018_TrafficForML_CICFl cumulative  2,097,150 rows |  2.38 GB free
  [ 3/10] Friday-23-02-2018_TrafficForML_CICFl cumulative  3,145,725 rows |  1.78 GB free

out of memory in file 4/10 (Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv)
  0.66 GB free, floor 0.7 GB
  [ 4/10] Thuesday-20-02-2018_TrafficForML_CIC cumulative  5,245,725 rows |  0.66 GB free

pandas held 5,245,725 of 16,233,002 rows (32.3%) in 2.0 min
stopped at: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv after 5,245,725 rows
memory released, 4.06 GB free


Now start Spark. Everything from here runs on the distributed engine.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("CA1-BigData-ETL")
    .master("local[*]")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version}, {spark.sparkContext.defaultParallelism} cores")
print(f"driver memory: {spark.conf.get('spark.driver.memory')}")

Spark 3.5.9, 4 cores
driver memory: 3g
